In [1]:
import os, sys, numpy as np
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", "/tmp")

repo_root = Path.cwd().resolve()
for candidate in [repo_root, *repo_root.parents]:
    if (candidate / "RL4CRN").exists() and (candidate / "apps").exists():
        repo_root = candidate
        break
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))


print("Python:", sys.version.split()[0])
print("CWD:", os.getcwd())
print("Repo root:", repo_root)


Python: 3.10.12
CWD: /local0/home/mfilo/git/GenAI-Net/apps
Repo root: /local0/home/mfilo/git/GenAI-Net


## 1) Import RL4CRN helpers


In [2]:
from RL4CRN.utils.input_interface import (
    Configurator,
    make_task,
    make_session_and_trainer,
    print_task_summary,
)
from RL4CRN.utils.default_tasks.DoseResponseTaskKind import DoseResponseTaskKind


## 2) Build a template IO/CRN


In [3]:
from RL4CRN.utils.crn_builders import build_simple_IOCRN

# choose preset
cfg = Configurator.preset("paper")

# select simulator and set tolerances
cfg.solver.algorithm = "CVODE"
cfg.solver.rtol = 1e-10
cfg.solver.atol = 1e-10

# build template IO/CRN
species_labels = ['X_1', 'X_2', 'X_3']
crn, species_labels = build_simple_IOCRN(
    species=species_labels,
    production_input_map={"X_1": "u_1", "X_2": "u_2"},
    degradation_input_map={},
    dilution_map={},
    output_species="X_3",
    solver=cfg.solver,
)

print("Template CRN built.")
print(" - num_inputs:", crn.num_inputs)
print(" - num_species:", len(species_labels))
print(" - species:", species_labels)


Template CRN built.
 - num_inputs: 2
 - num_species: 3
 - species: ['X_1', 'X_2', 'X_3']


## 3) Build the reaction library (MAK)


In [4]:
from RL4CRN.utils.library_builders import build_MAK_library

# library components
library_components = build_MAK_library(crn, species_labels, order=2)

library, M, K, masks = library_components
print("Library built.")
print(" - M (num reactions in library):", M)
print(" - K (num parameters in library):", K)


Library built.
 - M (num reactions in library): 91
 - K (num parameters in library): 91


## 4) Define the task: Dose Response


In [5]:
from RL4CRN.utils.input_interface import get_task_kind
get_task_kind("dose_response").pretty_help()

### TaskKind `dose_response`

**Required params**
- `target`: float OR callable with named args (recommended)
- `dose_range`: Tuple[u_min, u_max, n]

**Optional params**
- `t_f`: float
- `n_t`: int
- `ic`: IC spec
- `weights`: weights spec
- `u_list`: explicit u_list
- `u_spec`: ('custom'|'grid'|'linspace', ...) escape hatch
- `norm`: int (default 1)
- `LARGE_NUMBER`: float (default 1e4)

**Notes**
- Default u_list is 1D linspace over dose_range with vectors shape (1,). If target is callable, its
  arg names are resolved via input_idx_dict/species_idx_dict.


In [6]:
task = make_task(
    template_crn=crn,
    library_components=library_components,
    kind="dose_response",
    species_labels=species_labels,
    params={
        "t_f": 100,
        "n_t": 1000,
        "ic": ("constant", 0.01),
        "weights": "transient",
        "u_spec": ("grid", [0.1, 0.4, 0.7, 1.0]),
        "target": lambda u_1, u_2: u_1 / u_2,
    }
)

print_task_summary(task)

# --- Optional safety checks (recommended) ---
print("Sanity checks:")
print(" - template num_inputs:", crn.num_inputs)
print(" - first u shape:", np.asarray(task.u_list[0]).shape)
print(" - first u length:", len(task.u_list[0]))
assert len(task.u_list[0]) == crn.num_inputs, "Input dimension mismatch: u has wrong length!"


Task: dose_response
time_horizon: (1000,) [0..100.0]
num scenarios: 16
first 3 u: [array([0.1, 0.1], dtype=float32), array([0.1, 0.4], dtype=float32), array([0.1, 0.7], dtype=float32)]

Sanity checks:
 - template num_inputs: 2
 - first u shape: (2,)
 - first u length: 2


## 5) Training configuration

In [7]:
# ---- Train config ----
cfg.train.max_added_reactions = 4
cfg.train.epochs = 31
cfg.train.render_every = 1
cfg.train.seed = 0
cfg.train.hall_of_fame_size = 30
cfg.train.batch_size = 1280

cfg.agent.risk_scheduler = {'risk': 0.9, 'risk_update': 0.0, 'max_risk': 1.0, 'risk_schedule': 1000}
cfg.policy.entropy_weights_per_head = {"structure": 3.0, "continuous": 1.0, "discrete": 0.0, "input_influence": 0.0}

In [8]:
# ---- rendering ----
cfg.render.n_best = 10
cfg.render.disregarded_percentage = 0.9
cfg.render.mode = {  # Mode of the experiment
    'style': 'logger', 
    'task': 'transients', 
    'format': 'image',
    'topology': True
}

## 6) Inspect full configuration (optional)


In [9]:
cfg.describe()

{'task': None,
 'solver': {'algorithm': 'CVODE', 'rtol': 1e-10, 'atol': 1e-10},
 'train': {'epochs': 31,
           'max_added_reactions': 4,
           'render_every': 1,
           'hall_of_fame_size': 30,
           'batch_multiplier': 10,
           'seed': 0,
           'n_cpus': None,
           'batch_size': 1280},
 'policy': {'width': 1024,
            'depth': 5,
            'deep_layer_size': 10240,
            'continuous_distribution': {'type': 'lognormal_1D'},
            'entropy_weights_per_head': {'structure': 3.0, 'continuous': 1.0, 'discrete': 0.0, 'input_influence': 0.0},
            'ordering_enabled': False,
            'constraint_strength': inf,
            'zero_reaction_idx': None,
            'stop_flag': False},
 'agent': {'learning_rate': 0.0001,
           'entropy_scheduler': {'entropy_weight': 0.001,
                                 'topk_entropy_weight': 1.0,
                                 'remainder_entropy_weight': 1.0,
                              

## 7) Create session + trainer

This step wires together:
- parallel environments
- observer/tensorizer/actuator/stepper interfaces
- policy + agent
- the chosen task reward function

The returned object:
- `trainer`: runs rollout → reward eval → policy update loops


In [10]:
import os
from datetime import datetime
from pytorch_lightning.loggers import CometLogger

task_name = "DoseResponse_Division_Task"
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# Expect these in your environment:
#   COMET_API_KEY   (required)
#   COMET_WORKSPACE (required)
api_key = os.environ["COMET_API_KEY"]
workspace = os.environ["COMET_WORKSPACE"]

logger = CometLogger(
    api_key=api_key,
    project=task_name,
    workspace=workspace,
    name=f"{task_name}_{timestamp}",
)

logger = logger.experiment

COMET WARNING: To get all data logged automatically, import comet_ml before the following modules: torch.
COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/maurice-filo/doseresponse-division-task/ad572f66396b49fabbea71fbe83caa69



In [11]:
trainer = make_session_and_trainer(cfg, task, logger=logger)

## 8) Train and save checkpoints


In [12]:
checkpoint_path = "Division_task_chkpt.pkl"
trainer.run(epochs=cfg.train.epochs, checkpoint_path=checkpoint_path)


[cvHandleFailure, Error: -15] At t = 27.2471988152536, unable to satisfy inequality constraints.


[cvHandleFailure, Error: -15] At t = 28.954520243703, unable to satisfy inequality constraints.


[cvHandleFailure, Error: -15] At t = 9.54986057985854, unable to satisfy inequality constraints.


[cvHandleFailure, Error: -15] At t = 11.3248648823883, unable to satisfy inequality constraints.


[cvHandleFailure, Error: -15] At t = 15.7979042888503, unable to satisfy inequality constraints.


[cvHandleFailure, Error: -15] At t = 49.9997315712845, unable to satisfy inequality constraints.


[cvHandleFailure, Error: -15] At t = 7.3944520304905, unable to satisfy inequality constraints.


[cvHandleFailure, Error: -15] At t = 27.5992154587883, unable to satisfy inequality constraints.


[cvHandleFailure, Error: -15] At t = 23.4094738025385, unable to satisfy inequality constraints.


[cvHandleFailure, Error: -15] At t = 38.6583353472397, unable to satisfy inequality constraints.


[cvHandleFa

/local0/home/mfilo/git/GenAI-Net/RL4CRN/environments/environment.py:232: UserWarning: The figure layout has changed to tight
  fig.tight_layout(rect=[0, 0, 1, 0.95])
/local0/home/mfilo/git/GenAI-Net/RL4CRN/environments/environment.py:232: UserWarning: The figure layout has changed to tight
  fig.tight_layout(rect=[0, 0, 1, 0.95])
/local0/home/mfilo/git/GenAI-Net/RL4CRN/environments/environment.py:232: UserWarning: The figure layout has changed to tight
  fig.tight_layout(rect=[0, 0, 1, 0.95])
/local0/home/mfilo/git/GenAI-Net/RL4CRN/environments/environment.py:232: UserWarning: The figure layout has changed to tight
  fig.tight_layout(rect=[0, 0, 1, 0.95])
/local0/home/mfilo/git/GenAI-Net/RL4CRN/environments/environment.py:232: UserWarning: The figure layout has changed to tight
  fig.tight_layout(rect=[0, 0, 1, 0.95])
/local0/home/mfilo/git/GenAI-Net/RL4CRN/environments/environment.py:232: UserWarning: The figure layout has changed to tight
  fig.tight_layout(rect=[0, 0, 1, 0.95])
/loc

Saved checkpoint: Division_task_chkpt.pkl

[cvHandleFailure, Error: -15] At t = 28.8066686800439, unable to satisfy inequality constraints.


[cvHandleFailure, Error: -15] At t = 36.9672914268042, unable to satisfy inequality constraints.


[cvHandleFailure, Error: -15] At t = 22.0052527165282, unable to satisfy inequality constraints.


[cvHandleFailure, Error: -15] At t = 16.9656411884943, unable to satisfy inequality constraints.


[cvHandleFailure, Error: -15] At t = 13.2362773933833, unable to satisfy inequality constraints.


[cvHandleFailure, Error: -15] At t = 18.2826591984822, unable to satisfy inequality constraints.


[cvHandleFailure, Error: -15] At t = 7.87424537647208, unable to satisfy inequality constraints.


[cvHandleFailure, Error: -15] At t = 11.91910957193, unable to satisfy inequality constraints.


[cvHandleFailure, Error: -15] At t = 4.53085216916335, unable to satisfy inequality constraints.


[cvHandleFailure, Error: -15] At t = 6.30805030789761, unable to sat

/local0/home/mfilo/git/GenAI-Net/RL4CRN/environments/environment.py:232: UserWarning: The figure layout has changed to tight
  fig.tight_layout(rect=[0, 0, 1, 0.95])
/local0/home/mfilo/git/GenAI-Net/RL4CRN/environments/environment.py:232: UserWarning: The figure layout has changed to tight
  fig.tight_layout(rect=[0, 0, 1, 0.95])
/local0/home/mfilo/git/GenAI-Net/RL4CRN/environments/environment.py:232: UserWarning: The figure layout has changed to tight
  fig.tight_layout(rect=[0, 0, 1, 0.95])
/local0/home/mfilo/git/GenAI-Net/RL4CRN/environments/environment.py:232: UserWarning: The figure layout has changed to tight
  fig.tight_layout(rect=[0, 0, 1, 0.95])
/local0/home/mfilo/git/GenAI-Net/RL4CRN/environments/environment.py:232: UserWarning: The figure layout has changed to tight
  fig.tight_layout(rect=[0, 0, 1, 0.95])
/local0/home/mfilo/git/GenAI-Net/RL4CRN/environments/environment.py:232: UserWarning: The figure layout has changed to tight
  fig.tight_layout(rect=[0, 0, 1, 0.95])
/loc

Saved checkpoint: Division_task_chkpt.pkl

[cvHandleFailure, Error: -15] At t = 22.3879329613686, unable to satisfy inequality constraints.


[cvHandleFailure, Error: -15] At t = 32.5143312225283, unable to satisfy inequality constraints.


[cvHandleFailure, Error: -15] At t = 20.0435614685167, unable to satisfy inequality constraints.


[cvHandleFailure, Error: -15] At t = 15.3290617020401, unable to satisfy inequality constraints.


[cvHandleFailure, Error: -15] At t = 19.5130787879999, unable to satisfy inequality constraints.


[cvHandleFailure, Error: -15] At t = 16.0698345551819, unable to satisfy inequality constraints.


[cvHandleFailure, Error: -15] At t = 13.252906278551, unable to satisfy inequality constraints.


[cvHandleFailure, Error: -15] At t = 5.72455791712902, unable to satisfy inequality constraints.


[cvHandleFailure, Error: -15] At t = 10.3039844037008, unable to satisfy inequality constraints.


[cvHandleFailure, Error: -15] At t = 9.30998091403512, unable to sa

...

Saved checkpoint: Division_task_chkpt.pkl
[epoch 29] best loss=0.05116 | median loss=0.723


/local0/home/mfilo/git/GenAI-Net/RL4CRN/environments/environment.py:232: UserWarning: The figure layout has changed to tight
  fig.tight_layout(rect=[0, 0, 1, 0.95])
/local0/home/mfilo/git/GenAI-Net/RL4CRN/environments/environment.py:232: UserWarning: The figure layout has changed to tight
  fig.tight_layout(rect=[0, 0, 1, 0.95])
/local0/home/mfilo/git/GenAI-Net/RL4CRN/environments/environment.py:232: UserWarning: The figure layout has changed to tight
  fig.tight_layout(rect=[0, 0, 1, 0.95])
/local0/home/mfilo/git/GenAI-Net/RL4CRN/environments/environment.py:232: UserWarning: The figure layout has changed to tight
  fig.tight_layout(rect=[0, 0, 1, 0.95])
/local0/home/mfilo/git/GenAI-Net/RL4CRN/environments/environment.py:232: UserWarning: The figure layout has changed to tight
  fig.tight_layout(rect=[0, 0, 1, 0.95])
/local0/home/mfilo/git/GenAI-Net/RL4CRN/environments/environment.py:232: UserWarning: The figure layout has changed to tight
  fig.tight_layout(rect=[0, 0, 1, 0.95])
/loc

Saved checkpoint: Division_task_chkpt.pkl
[epoch 30] best loss=0.05126 | median loss=0.4476


/local0/home/mfilo/git/GenAI-Net/RL4CRN/environments/environment.py:232: UserWarning: The figure layout has changed to tight
  fig.tight_layout(rect=[0, 0, 1, 0.95])
/local0/home/mfilo/git/GenAI-Net/RL4CRN/environments/environment.py:232: UserWarning: The figure layout has changed to tight
  fig.tight_layout(rect=[0, 0, 1, 0.95])
/local0/home/mfilo/git/GenAI-Net/RL4CRN/environments/environment.py:232: UserWarning: The figure layout has changed to tight
  fig.tight_layout(rect=[0, 0, 1, 0.95])
/local0/home/mfilo/git/GenAI-Net/RL4CRN/environments/environment.py:232: UserWarning: The figure layout has changed to tight
  fig.tight_layout(rect=[0, 0, 1, 0.95])
/local0/home/mfilo/git/GenAI-Net/RL4CRN/environments/environment.py:232: UserWarning: The figure layout has changed to tight
  fig.tight_layout(rect=[0, 0, 1, 0.95])
/local0/home/mfilo/git/GenAI-Net/RL4CRN/environments/environment.py:232: UserWarning: The figure layout has changed to tight
  fig.tight_layout(rect=[0, 0, 1, 0.95])
/loc

Saved checkpoint: Division_task_chkpt.pkl


## 9) Inspect the best CRN

The trainer keeps a **Hall of Fame** of good CRNs found during rollouts.


In [ ]:
trainer.inspect_best(plot=True)

best = trainer.best_crn()
print("Hall of Fame size:", len(trainer.s.mult_env.hall_of_fame))
if best is not None:
    print("Best loss:", best.last_task_info.get("reward", None))

## 10) Sample and re-simulate


In [ ]:
trainer.sample(10, 10, ic=("constant", 1.0))

We can now inspect newly sampled I/O CRNs.

In [ ]:
import matplotlib.pyplot as plt

index = 0
crn_s = trainer.get_sampled_crns()[index]
print(crn_s)
print("reward:", crn_s.last_task_info.get("reward", None))

# Plotters depend on your IOCRN implementation
crn_s.plot_transient_response(); plt.show()


Save again our results.

In [ ]:
trainer.save(checkpoint_path)

## 11) Loading a saved Session/Trainer from a checkpoint


In [ ]:
from RL4CRN.utils.input_interface import load_session_and_trainer

trainer_loaded = load_session_and_trainer(checkpoint_path, device="cuda")
trainer_loaded.inspect_best()

## 12) Re-simulate Hall-of-Fame CRNs under new conditions


In [ ]:
hof_crns = [item.state for item in trainer.s.mult_env.hall_of_fame]

trainer.s.crn_template

crns_new = trainer.resimulate(
    hof_crns,
    ic=("constant", 0.4), 
    u_spec=("grid", [0.0, 1.0]),
)

trainer.inspect(crns_new[0])
crns_new[0].plot_transient_response(); plt.show()
